In [ ]:
import numpy as np
import pandas as pd
import sys
import os
import pickle
from collections import defaultdict
import json
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append("../experiments")
import experiment_utils
sys.path.append("../io-bots-irl/irl")
import data_corruption_utils

In [ ]:
with open("../../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]


In [ ]:
traj_df = pd.read_pickle(dataframes_path + "/sampled_matched_perturbed_df_final.pkl")

In [ ]:
for dropped_percent in traj_df.perturb_percent.unique():
    running_confusion_matrix_rf = np.zeros((2, 2), dtype=int)
    running_confusion_matrix_xgb = np.zeros((2, 2), dtype=int)

    f1_list_rf = []
    f1_list_xgb = []

    misclassified_users_rf_all_subs = defaultdict(int)
    misclassified_users_xgb_all_subs = defaultdict(int)
    for run in traj_df.run.unique():
        
        print(dropped_percent,"run:",run)
        seed = 100 + run

        df_troll_sampled = traj_df[(traj_df.run == run) & (traj_df.perturb_percent == dropped_percent) & (traj_df.russian == 1)].copy()
        df_organic_sampled = traj_df[(traj_df.run == run) & (traj_df.perturb_percent == dropped_percent) & (traj_df.russian == 0)].copy()

        df_troll_sampled["feature_col"] = df_troll_sampled['traj_counts_perturbed'].apply(data_corruption_utils.normalize_replace_zeros)
        df_organic_sampled["feature_col"] = df_organic_sampled['traj_counts_perturbed'].apply(data_corruption_utils.normalize_replace_zeros)

        confusion_matrix_rf, confusion_matrix_xgb, overall_rf_f1, overall_xgb_f1, misclassified_users_rf, misclassified_users_xgb = experiment_utils.run_experiment(df_troll_sampled,df_organic_sampled,seed = seed)

        f1_list_rf.append(overall_rf_f1)
        f1_list_xgb.append(overall_xgb_f1)
        running_confusion_matrix_rf += confusion_matrix_rf
        running_confusion_matrix_xgb += confusion_matrix_xgb
        for user in misclassified_users_rf:
            misclassified_users_rf_all_subs[user] += misclassified_users_rf[user]
        for user in misclassified_users_xgb:
            misclassified_users_xgb_all_subs[user] += misclassified_users_xgb[user]

    print("dropped_percent", dropped_percent)
    print("Final Confusion Matrix (Random Forest):\n", running_confusion_matrix_rf)    
    f1 = np.mean(f1_list_rf)
    print(f"Mean F1 Score (Random Forest): {f1:.2f}\n")

    print("Final Confusion Matrix (XGBoost):\n", running_confusion_matrix_xgb)        
    f1 = np.mean(f1_list_xgb)
    print(f"Mean F1 Score (XGBoost): {f1:.2f}\n")


    results = {
        "f1_list_rf":f1_list_rf,
        "f1_list_xgb":f1_list_xgb,
        "running_confusion_matrix_rf":running_confusion_matrix_rf,
        "running_confusion_matrix_xgb":running_confusion_matrix_xgb,
        "misclassified_users_rf_all_subs":misclassified_users_rf_all_subs,
        "misclassified_users_xgb_all_subs":misclassified_users_xgb_all_subs
    }


    output_dir = project_output_path + "/perturb/normalised_traj_count"
    os.makedirs(output_dir, exist_ok=True)
    with open(output_dir +'/' + str(dropped_percent) + '_results.pkl', 'wb') as f:
        pickle.dump(results, f)

0.0 run: 0
Confusion Matrix (Random Forest):
[[93  6]
 [ 5 94]]
Confusion Matrix (XGBoost):
[[91  8]
 [ 4 95]]
F1 Score (Random Forest): 0.94
F1 Score (XGBoost): 0.94


0.0 run: 1
Confusion Matrix (Random Forest):
[[93  6]
 [ 4 95]]
Confusion Matrix (XGBoost):
[[94  5]
 [ 6 93]]
F1 Score (Random Forest): 0.95
F1 Score (XGBoost): 0.94


0.0 run: 2
Confusion Matrix (Random Forest):
[[93  6]
 [ 5 94]]
Confusion Matrix (XGBoost):
[[94  5]
 [ 4 95]]
F1 Score (Random Forest): 0.94
F1 Score (XGBoost): 0.95


0.0 run: 3
Confusion Matrix (Random Forest):
[[95  4]
 [ 8 91]]
Confusion Matrix (XGBoost):
[[92  7]
 [ 7 92]]
F1 Score (Random Forest): 0.94
F1 Score (XGBoost): 0.93


0.0 run: 4
Confusion Matrix (Random Forest):
[[97  2]
 [ 3 96]]
Confusion Matrix (XGBoost):
[[95  4]
 [ 5 94]]
F1 Score (Random Forest): 0.97
F1 Score (XGBoost): 0.95


0.0 run: 5
Confusion Matrix (Random Forest):
[[90  9]
 [ 7 92]]
Confusion Matrix (XGBoost):
[[90  9]
 [ 8 91]]
F1 Score (Random Forest): 0.92
F1 Score (XGB